In [ ]:
!pip install matplotlib-venn
!pip install datasets transformers evaluate
!pip install sentence-transformers
!pip install fsspec==2023.6.0
!pip install rouge_score
import torch
from transformers import (
    LEDTokenizer,
    LEDForConditionalGeneration,
    get_linear_schedule_with_warmup
)
from datasets import load_dataset
from torch.optim import AdamW
from torch.utils.data import DataLoader
from tqdm import tqdm
import evaluate
import pandas as pd


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


def prepare_cnndm_dataset():
    dataset = load_dataset("cnn_dailymail", "3.0.0")

    def preprocess_function(examples):
        inputs = ["summarize: " + doc for doc in examples["article"]]
        targets = examples["highlights"]
        return {"inputs": inputs, "targets": targets}

    dataset = dataset.map(preprocess_function, batched=True)
    return dataset["train"], dataset["validation"], dataset["test"]

train_dataset, val_dataset, test_dataset = prepare_cnndm_dataset()

model_name = "allenai/led-base-16384"
tokenizer = LEDTokenizer.from_pretrained(model_name)
model = LEDForConditionalGeneration.from_pretrained(model_name).to(device)

def data_collator(batch):
    inputs = [item["inputs"] for item in batch]
    targets = [item["targets"] for item in batch]

    inputs_tokenized = tokenizer(
        inputs,
        max_length=4096,
        truncation=True,
        padding="max_length",
        return_tensors="pt"
    )

    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            targets,
            max_length=256,
            truncation=True,
            padding="max_length",
            return_tensors="pt"
        ).input_ids
        labels[labels == tokenizer.pad_token_id] = -100

    inputs_tokenized["labels"] = labels
    return {k: v.to(device) for k, v in inputs_tokenized.items()}

def train_model(model, train_dataset, val_dataset, epochs=1, batch_size=2):
    train_loader = DataLoader(
        train_dataset.select(range(500)),  # partial for CPU
        batch_size=batch_size,
        shuffle=True,
        collate_fn=data_collator
    )

    optimizer = AdamW(model.parameters(), lr=5e-5)
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=50,
        num_training_steps=len(train_loader) * epochs
    )

    model.train()
    for epoch in range(epochs):
        total_loss = 0.0
        progress = tqdm(train_loader, desc=f"Epoch {epoch+1}")
        for batch in progress:
            outputs = model(**batch)
            loss = outputs.loss
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            total_loss += loss.item()
        print(f"Epoch {epoch+1} Loss: {total_loss/len(train_loader):.4f}")

    return model

model = train_model(model, train_dataset, val_dataset)

rouge = evaluate.load("rouge")

def evaluate_model(model, test_dataset, num_samples=50):
    test_subset = test_dataset.select(range(num_samples))
    predictions, references = [], []

    model.eval()
    with torch.no_grad():
        for example in tqdm(test_subset, desc="Evaluating"):
            input_ids = tokenizer(
                example["inputs"],
                max_length=4096,
                truncation=True,
                padding="max_length",
                return_tensors="pt"
            ).input_ids.to(device)

            attention_mask = input_ids.ne(tokenizer.pad_token_id).long().to(device)
            global_attention_mask = torch.zeros_like(attention_mask)
            global_attention_mask[:, 0] = 1  # first token is always global

            summary_ids = model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                global_attention_mask=global_attention_mask,
                max_length=256,
                num_beams=4
            )

            prediction = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
            predictions.append(prediction)
            references.append(example["targets"])

    return rouge.compute(predictions=predictions, references=references, use_stemmer=True)

results = evaluate_model(model, test_dataset)

results_df = pd.DataFrame({
    "Model": ["LED-base-16384"],
    "ROUGE-1": [results["rouge1"]],
    "ROUGE-2": [results["rouge2"]],
    "ROUGE-L": [results["rougeL"]]
})
print("\nEvaluation Results:")
print(results_df)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 113.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 93.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 58.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 79.5 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstallin

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Extracting data files:   0%|          | 0/3 [00:00<?, ?it/s]

Generating test split:   0%|          | 0/11490 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/287113 [00:00<?, ? examples/s]

...

generation_config.json:   0%|          | 0.00/168 [00:00<?, ?B/s]


Epoch 1:   0%|          | 0/250 [00:00<?, ?it/s]/usr/local/lib/python3.11/dist-packages/transformers/tokenization_utils_base.py:3951: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(

Epoch 1:   0%|          | 1/250 [00:04<18:45,  4.52s/it]/usr/local/lib/python3.11/dist-packages/transformers/tokenization_utils_base.py:3951: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(

Epoch 1:   1%|          | 2/250 [00:07<14:36,  3.53s/it]/usr/local/lib/python3.11/dist-packag

Epoch 1 Loss: 2.3837


Evaluating: 100%|██████████| 50/50 [00:53<00:00,  1.08s/it]



Evaluation Results:
            Model  ROUGE-1   ROUGE-2   ROUGE-L
0  LED-base-16384   0.2775  0.116264  0.210546
